# Neural Oblivious Decision Ensembles
5-fold stratified cross-validation on the diabetes readmission dataset.

In [7]:
import os
import sys

# Repo root: makes Preprocessing.* and FeatureEngineering.* importable
sys.path.insert(0, os.path.abspath('..'))
# Preprocessing/ directly: needed because encode.py has bare 'from data_loading import *'
sys.path.insert(0, os.path.abspath('../Preprocessing'))
# node-master/: makes 'import lib' work
sys.path.insert(0, os.path.abspath('../node-master'))

In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import deque
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import roc_auc_score

import lib
from Preprocessing.data_loading import read_data
from Preprocessing.encode import full_reduce_df, full_ordinal_encode, full_one_hot_encode
from FeatureEngineering.diagnosis_graph import DiagnosisGraph

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


In [10]:
NUM_FOLDS = 10      # uses pre-defined fold column in diabetes_stratified.csv
LAYER_DIM = 256
NUM_LAYERS = 2
TREE_DIM = 3
DEPTH = 6
BATCH_SIZE = 512
LR = 1e-3
N_CHECKPOINTS = 5
REPORT_EVERY = 50
EARLY_STOPPING_ROUNDS = 500
RANDOM_STATE = 42

In [11]:
TARGET_COL = 'readmitted_30_days'
FOLD_COL = 'fold'

df = read_data('diabetes_stratified', 'Stratified')
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COL].value_counts()}")
print(f"Folds: {sorted(df[FOLD_COL].unique())}")

Shape: (71518, 49)
Target distribution:
readmitted_30_days
0    65225
1     6293
Name: count, dtype: int64
Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [12]:
# Build (train_indices, valid_indices) pairs from the pre-defined fold column
fold_indices = [
    (df[df[FOLD_COL] != k].index.tolist(), df[df[FOLD_COL] == k].index.tolist())
    for k in range(NUM_FOLDS)
]
print(f"Fold sizes: {[(len(tr), len(va)) for tr, va in fold_indices]}")

Fold sizes: [(64365, 7153), (64365, 7153), (64365, 7153), (64366, 7152), (64366, 7152), (64367, 7151), (64367, 7151), (64367, 7151), (64367, 7151), (64367, 7151)]


In [13]:
def preprocess_fold(df_train, df_valid):
    """Full preprocessing pipeline for one fold. All steps fit on train only."""
    y_train = df_train[TARGET_COL].values.astype(np.float32)
    y_valid = df_valid[TARGET_COL].values.astype(np.float32)
    X_train = df_train.drop(columns=[TARGET_COL, FOLD_COL])
    X_valid = df_valid.drop(columns=[TARGET_COL, FOLD_COL])

    # Drop 100%-pure columns based on train
    X_train, X_valid = full_reduce_df(X_train, X_valid)

    # Ordinal encode age/weight to numeric midpoints
    X_train, X_valid = full_ordinal_encode(X_train, X_valid)

    # DiagnosisGraph: fit on train, drops diag_1/2/3, adds ~100 graph features
    graph = DiagnosisGraph()
    X_train, X_valid = graph.fit_transform_fold_augmented(
        X_train, X_valid, drop_diag_cols=True, reset_index=True
    )

    # One-hot encode nominal columns, fitted on train
    X_train, X_valid = full_one_hot_encode(X_train, X_valid)

    # Drop non-numeric columns (e.g. weight='Unknown' stays as string after ordinal encode)
    numeric_cols = X_train.select_dtypes(include='number').columns
    X_train = X_train[numeric_cols]
    X_valid = X_valid[numeric_cols]

    # QuantileTransformer: fit on train, normalize to standard normal
    qt = QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)
    X_train_np = qt.fit_transform(X_train.values).astype(np.float32)
    X_valid_np = qt.transform(X_valid.values).astype(np.float32)

    return X_train_np, y_train, X_valid_np, y_valid

In [14]:
# Verify preprocessing on fold 0
train_idx_f0, valid_idx_f0 = fold_indices[0]
df_train_f0 = df.loc[train_idx_f0].reset_index(drop=True)
df_valid_f0 = df.loc[valid_idx_f0].reset_index(drop=True)

X_tr, y_tr, X_va, y_va = preprocess_fold(df_train_f0, df_valid_f0)

print(f"X_train shape: {X_tr.shape}, dtype: {X_tr.dtype}")
print(f"X_valid shape: {X_va.shape}, dtype: {X_va.dtype}")
print(f"y_train unique values: {np.unique(y_tr)}, positive rate: {y_tr.mean():.3f}")
print(f"NaN in X_train: {np.isnan(X_tr).sum()}")
print(f"NaN in X_valid: {np.isnan(X_va).sum()}")

X_train shape: (64365, 282), dtype: float32
X_valid shape: (7153, 282), dtype: float32
y_train unique values: [0. 1.], positive rate: 0.088
NaN in X_train: 0
NaN in X_valid: 0


In [15]:
def build_model(in_features):
    return nn.Sequential(
        lib.DenseBlock(
            in_features,
            layer_dim=LAYER_DIM,
            num_layers=NUM_LAYERS,
            tree_dim=TREE_DIM,
            depth=DEPTH,
            flatten_output=False,
            choice_function=lib.entmax15,
            bin_function=lib.entmoid15,
        ),
        lib.Lambda(lambda x: x[..., 0].mean(dim=-1)),  # mean over trees → scalar logit
    ).to(device)


def init_model(model, X_train_np):
    """Run forward pass on first 1000 samples to trigger data-aware threshold init."""
    with torch.no_grad():
        model(torch.as_tensor(X_train_np[:1000], device=device))
    return model

In [16]:
# Verify model builds, initialises, and produces finite scalar output
_test_model = build_model(X_tr.shape[1])
_test_model = init_model(_test_model, X_tr)

with torch.no_grad():
    _out = _test_model(torch.as_tensor(X_tr[:4], device=device))

print(f"Output shape: {_out.shape}")          # expected: torch.Size([4])
print(f"Sample logits: {_out.cpu().numpy()}")  # expected: finite floats
del _test_model

Output shape: torch.Size([4])
Sample logits: [-0.00068815  0.00227828 -0.00085677  0.0035662 ]


In [17]:
def predict_proba(model, X_np, batch_size=512):
    """Return sigmoid probabilities for binary classification."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            batch = torch.as_tensor(X_np[i:i + batch_size], device=device)
            preds.append(torch.sigmoid(model(batch)).cpu().numpy())
    return np.concatenate(preds)


def average_state_dicts(state_dicts):
    """Element-wise mean of a list of model state dicts."""
    avg = {}
    for key in state_dicts[0]:
        avg[key] = torch.stack([sd[key].float() for sd in state_dicts]).mean(dim=0)
    return avg

In [18]:
def train_fold(X_train_np, y_train_np, X_valid_np, y_valid_np):
    """Train NODE on one fold. Returns (best_val_auc, best_state_dict)."""
    model = build_model(X_train_np.shape[1])
    model = init_model(model, X_train_np)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    checkpoints = deque(maxlen=N_CHECKPOINTS)
    best_auc = 0.0
    best_state = None
    patience = 0
    max_patience = EARLY_STOPPING_ROUNDS // REPORT_EVERY
    step = 0

    for X_batch, y_batch in lib.iterate_minibatches(
        X_train_np, y_train_np, batch_size=BATCH_SIZE, shuffle=True, epochs=float('inf')
    ):
        model.train()
        x_t = torch.as_tensor(X_batch, device=device)
        y_t = torch.as_tensor(y_batch, device=device)

        optimizer.zero_grad()
        criterion(model(x_t), y_t).backward()
        optimizer.step()
        step += 1

        if step % REPORT_EVERY == 0:
            checkpoints.append(deepcopy(model.state_dict()))
            avg_state = average_state_dicts(list(checkpoints))

            model.load_state_dict(avg_state)
            auc = roc_auc_score(y_valid_np, predict_proba(model, X_valid_np))

            if auc > best_auc:
                best_auc = auc
                best_state = deepcopy(avg_state)
                patience = 0
            else:
                patience += 1

            model.load_state_dict(checkpoints[-1])  # restore latest for next training step

            if patience >= max_patience:
                print(f"  Early stop at step {step}, best AUC: {best_auc:.4f}")
                break

    return best_auc, best_state

In [19]:
# Smoke test: short run on fold 0 to verify the loop runs end-to-end
_saved = EARLY_STOPPING_ROUNDS
EARLY_STOPPING_ROUNDS = 100

_auc_smoke, _ = train_fold(X_tr, y_tr, X_va, y_va)
print(f"Smoke test AUC: {_auc_smoke:.4f}")  # should be > 0.5

EARLY_STOPPING_ROUNDS = _saved

  Early stop at step 4500, best AUC: 0.6614
Smoke test AUC: 0.6614


In [ ]:
fold_aucs = []

for fold_num, (train_idx, valid_idx) in enumerate(fold_indices):
    print(f"\n=== Fold {fold_num + 1}/{NUM_FOLDS} ===")

    df_train_fold = df.loc[train_idx].reset_index(drop=True)
    df_valid_fold = df.loc[valid_idx].reset_index(drop=True)

    X_train_np, y_train_np, X_valid_np, y_valid_np = preprocess_fold(df_train_fold, df_valid_fold)
    print(f"  Features: {X_train_np.shape[1]}, Train: {len(X_train_np)}, Valid: {len(X_valid_np)}")

    best_auc, _ = train_fold(X_train_np, y_train_np, X_valid_np, y_valid_np)
    fold_aucs.append(best_auc)
    print(f"  Fold {fold_num + 1} AUC: {best_auc:.4f}")


=== Fold 1/10 ===
  Features: 282, Train: 64365, Valid: 7153
  Early stop at step 5100, best AUC: 0.6642
  Fold 1 AUC: 0.6642

=== Fold 2/10 ===
  Features: 282, Train: 64365, Valid: 7153
  Early stop at step 4200, best AUC: 0.6664
  Fold 2 AUC: 0.6664

=== Fold 3/10 ===
  Features: 282, Train: 64365, Valid: 7153
  Early stop at step 3350, best AUC: 0.6589
  Fold 3 AUC: 0.6589

=== Fold 4/10 ===
  Features: 283, Train: 64366, Valid: 7152
  Early stop at step 5000, best AUC: 0.6575
  Fold 4 AUC: 0.6575

=== Fold 5/10 ===
  Features: 283, Train: 64366, Valid: 7152
  Early stop at step 4350, best AUC: 0.6288
  Fold 5 AUC: 0.6288

=== Fold 6/10 ===
  Features: 283, Train: 64367, Valid: 7151


In [ ]:
fold_aucs_arr = np.array(fold_aucs)
print("\n=== NODE Cross-Validation Results ===")
for i, auc in enumerate(fold_aucs_arr):
    print(f"  Fold {i + 1}: {auc:.4f}")
print(f"\n  Mean AUC: {fold_aucs_arr.mean():.4f} ± {fold_aucs_arr.std():.4f}")